<a href="https://colab.research.google.com/github/sabithakrishnan/multimodal-image-analysis/blob/main/multimodalimage_recognition_lora.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install open-clip-torch datasets pillow torch torchvision matplotlib


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 19.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 3.2 MB/s eta 0:00:00


In [2]:

import os
from pathlib import Path
from datasets import load_dataset
from PIL import Image
from tqdm import tqdm

class ProductionDatasetBuilder:
    def __init__(self, dataset_repo: str = "blanchon/EuroSAT_RGB"):
        self.repo = dataset_repo

    def build_local_directory(self, output_dir: str = "satellite_dataset"):
        dest_path = Path(output_dir)

        # Guard clause: avoid re-running if data exists
        if dest_path.exists() and any(dest_path.iterdir()):
            print(f"📦 Workspace directory '{output_dir}' already exists. Transitioning steps...")
            return

        print(f"📡 Accessing Hugging Face native parquet tables for '{self.repo}'...")
        # Loads clean tabular shards; completely bypasses legacy python execution scripts
        dataset_split = load_dataset(self.repo, split="train")

        # Dynamically infer the land-use taxonomy from the Metadata Table schema
        label_names = dataset_split.features["label"].names
        print(f"🎯 Verified {len(label_names)} target categories for extraction.")

        print("\n💾 Unpacking and storing satellite tensors to local filesystems...")
        for idx, sample in enumerate(tqdm(dataset_split, desc="Processing Satellite Tiles")):
            pil_img = sample["image"]
            label_id = sample["label"]

            # Sanitize naming structures to eliminate runtime mapping mistakes
            class_name = label_names[label_id].replace(" ", "")

            # Map path structure: satellite_dataset/IndustrialBuildings/tile_x.jpg
            class_folder = dest_path / class_name
            class_folder.mkdir(parents=True, exist_ok=True)

            file_name = class_folder / f"tile_{idx}.jpg"

            if pil_img.mode != "RGB":
                pil_img = pil_img.convert("RGB")

            pil_img.save(file_name, "JPEG")

        print(f"\n✅ Step 2 Complete! File mapping successfully resolved inside: '{dest_path.resolve()}'")

if __name__ == "__main__":
    builder = ProductionDatasetBuilder()
    builder.build_local_directory()


📡 Accessing Hugging Face native parquet tables for 'blanchon/EuroSAT_RGB'...


README.md:   0%|          | 0.00/3.38k [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  105MB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 34.8MB            

...

Generating test split:   0%|          | 0/5400 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/5400 [00:00<?, ? examples/s]

🎯 Verified 10 target categories for extraction.

💾 Unpacking and storing satellite tensors to local filesystems...


Processing Satellite Tiles: 100%|██████████| 16200/16200 [00:26<00:00, 616.59it/s]


✅ Step 2 Complete! File mapping successfully resolved inside: '/content/satellite_dataset'


In [4]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
import open_clip
from peft import LoraConfig, get_peft_model
from tqdm import tqdm
from pathlib import Path
from torchvision.datasets import ImageFolder

!pip install --upgrade torchao

class LoRASatelliteClassifier(nn.Module):
    def __init__(self, clip_model, num_classes: int = 10):
        super().__init__()
        # Isolate the underlying Vision Transformer Tower from CLIP
        self.vision_encoder = clip_model.visual

        # Configure LoRA to hook into the self-attention projection matrices
        # In OpenCLIP ViT architectures, the attention projection layer is typically 'out_proj' or 'in_proj'
        lora_config = LoraConfig(
            r=16,
            lora_alpha=32,
            target_modules=["out_proj", "in_proj"],
            lora_dropout=0.05,
            bias="none"
        )

        # Wrap the vision encoder with LoRA adapters
        self.peft_vision_encoder = get_peft_model(self.vision_encoder, lora_config)

        # Add a standard classification head on top
        self.classifier_head = nn.Linear(512, num_classes)

    def forward(self, x):
        embeddings = self.peft_vision_encoder(x)
        embeddings = embeddings / embeddings.norm(dim=-1, keepdim=True)

        return self.classifier_head(embeddings)

def main():
    device = "cuda" if torch.cuda.is_available() else "cpu"
    data_directory = "satellite_dataset"
    batch_size = 32
    epochs = 5
    lr = 3e-4

    #  Load Core Model and Visual Transforms
    base_model, _, preprocess = open_clip.create_model_and_transforms(
        'ViT-B-32', pretrained='laion2b_s34b_b79k', device=device)

    # Setup Dataset Split
    full_dataset = ImageFolder(root=data_directory, transform=preprocess)
    num_classes = len(full_dataset.classes)

    train_size = int(0.8 * len(full_dataset))
    val_size = len(full_dataset) - train_size
    train_dataset, val_dataset = random_split(full_dataset, [train_size, val_size])

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=2)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=2)

    # Instantiate LoRA Network Architecture,optimise and schedule
    model = LoRASatelliteClassifier(base_model, num_classes=num_classes).to(device)
    model.peft_vision_encoder.print_trainable_parameters()
    optimizer = optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=lr, weight_decay=0.01)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    criterion = nn.CrossEntropyLoss().to(device)

    # Fine-Tuning Execution Loop
    print(f"\n🚀 Initiating Vision Tower Parameter Fine-Tuning on {device}...")
    for epoch in range(epochs):
        model.train()
        running_loss = 0.0

        for images, labels in tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}"):
            images, labels = images.to(device), labels.to(device)

            optimizer.zero_grad()
            logits = model(images)
            loss = criterion(logits, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item()

        scheduler.step()

        # Validation Pass
        model.eval()
        correct = 0
        total = 0
        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(device), labels.to(device)
                outputs = model(images)
                predictions = outputs.argmax(dim=-1)
                correct += (predictions == labels).sum().item()
                total += labels.size(0)

        epoch_acc = (correct / total) * 100
        print(f"📉 Epoch {epoch+1:02d} Loss: {running_loss/len(train_loader):.4f} | ✨ Val Accuracy: {epoch_acc:.2f}%")

    # 6. Save the Adapted Weights Locally
    output_dir = Path("lora_satellite_weights")
    output_dir.mkdir(exist_ok=True)
    torch.save(model.state_dict(), output_dir / "lora_clip_classifier.pt")
    print(f"\n💾 Model state successfully exported to {output_dir}/lora_clip_classifier.pt")

if __name__ == "__main__":
    main()

trainable params: 294,912 || all params: 88,144,128 || trainable%: 0.3346

🚀 Initiating Vision Tower Parameter Fine-Tuning on cpu...


Epoch 1/5: 100%|██████████| 405/405 [44:24<00:00,  6.58s/it]


📉 Epoch 01 Loss: 2.0982 | ✨ Val Accuracy: 66.33%


Epoch 2/5: 100%|██████████| 405/405 [43:50<00:00,  6.50s/it]


📉 Epoch 02 Loss: 1.7818 | ✨ Val Accuracy: 72.44%


...

📉 Epoch 03 Loss: 1.5821 | ✨ Val Accuracy: 75.06%


Epoch 4/5: 100%|██████████| 405/405 [43:24<00:00,  6.43s/it]


📉 Epoch 04 Loss: 1.4741 | ✨ Val Accuracy: 75.83%


Epoch 5/5: 100%|██████████| 405/405 [44:26<00:00,  6.58s/it]


📉 Epoch 05 Loss: 1.4301 | ✨ Val Accuracy: 76.17%

💾 Model state successfully exported to lora_satellite_weights/lora_clip_classifier.pt
